# 03 — Algorithm and Model Development

This notebook implements and compares two machine learning approaches for predicting AKI onset within a 6-hour horizon:

1. **Logistic Regression** — a linear baseline providing interpretable coefficients and clinical transparency
2. **XGBoost** — a gradient-boosted tree ensemble capable of capturing non-linear feature interactions

Both models are benchmarked against the **KDIGO rule-based criteria**, which serves as the clinical standard of care.

> **Modules Used**: `src.data_loader`, `src.features`, `src.evaluation`, `src.tuning`

In [1]:
import sys
sys.path.insert(0, '..')

from src.data_loader import load_gold_table, train_test_split_by_stay
from src.features import prepare_features, get_feature_columns
from src.evaluation import compute_metrics, kdigo_baseline_predictions, compare_models
from src.tuning import cross_validate_model

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import numpy as np
import pandas as pd

## 3.1 Data Loading and Patient-Level Split

The dataset is split into 80% training and 20% test sets using `GroupShuffleSplit`, grouped by `stay_id`. This ensures that **all hourly observations from a single ICU stay appear exclusively in one set** and prevents data leakage from temporally correlated rows.

In [2]:
df = load_gold_table()
train_df, test_df = train_test_split_by_stay(df)

Loaded 2,304,561 hourly samples across 72,082 stays.
Train: 1,842,484 samples (57,665 stays)
Test:  462,077 samples (14,417 stays)


## 3.3 Feature Preparation

In [3]:
# Logistic Regression: scaled features
X_train_lr, y_train, scaler = prepare_features(train_df, scale=True)
X_test_lr, y_test, _ = prepare_features(test_df, scale=True, scaler=scaler)

# XGBoost: raw features (no scaling needed)
X_train_xgb, _, _ = prepare_features(train_df, scale=False)
X_test_xgb, _, _ = prepare_features(test_df, scale=False)

print(f'Training samples: {len(X_train_lr):,} | Test samples: {len(X_test_lr):,}')
print(f'Features: {len(get_feature_columns())}')
print(f'Positive rate (train): {y_train.mean():.3%}')
print(f'Positive rate (test):  {y_test.mean():.3%}')

Training samples: 1,842,484 | Test samples: 462,077
Features: 18
Positive rate (train): 9.583%
Positive rate (test):  9.568%


## 3.4 KDIGO Baseline

In [4]:
kdigo_preds = kdigo_baseline_predictions(test_df)
print(f'KDIGO predictions — all zeros: {(kdigo_preds == 0).all()}')
print(f'KDIGO recall: {(kdigo_preds[y_test == 1] == 1).mean():.3f}')
print(f'\nThis confirms KDIGO has ZERO predictive lead time in our evaluation framework.')

KDIGO predictions — all zeros: True
KDIGO recall: 0.000

This confirms KDIGO has ZERO predictive lead time in our evaluation framework.


## 3.5 Logistic Regression — Training

In [5]:
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver='lbfgs',   
    C=1.0,
    random_state=42
)
lr_model.fit(X_train_lr, y_train)

lr_probs = lr_model.predict_proba(X_test_lr)[:, 1]
lr_metrics = compute_metrics(y_test, lr_probs)

print(f"Logistic Regression (default params):")
print(f"  AUPRC: {lr_metrics['auprc']:.4f}")
print(f"  AUROC: {lr_metrics['auroc']:.4f}")
print(f"  F1:    {lr_metrics['f1']:.4f}")


Logistic Regression (default params):
  AUPRC: 0.2351
  AUROC: 0.7546
  F1:    0.2883


## 3.6 XGBoost — Training

In [6]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / max(pos_count, 1)

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_xgb, y_train)

xgb_probs = xgb_model.predict_proba(X_test_xgb)[:, 1]
xgb_metrics = compute_metrics(y_test, xgb_probs)

print(f"XGBoost (default params):")
print(f"  AUPRC: {xgb_metrics['auprc']:.4f}")
print(f"  AUROC: {xgb_metrics['auroc']:.4f}")
print(f"  F1:    {xgb_metrics['f1']:.4f}")

XGBoost (default params):
  AUPRC: 0.3967
  AUROC: 0.8121
  F1:    0.3448


## 3.7 Cross-Validation (5-Fold GroupKFold)

To obtain robust performance estimates with confidence intervals, we run 5-fold cross-validation using `GroupKFold` grouped by `stay_id`. This ensures that each fold maintains patient-level separation, and the mean ± standard deviation across folds provides a measure of model stability.

In [7]:
# Cross-validate Logistic Regression
lr_cv_model = LogisticRegression(
    class_weight='balanced', max_iter=1000, solver='lbfgs', random_state=42
)
lr_cv = cross_validate_model(
    lr_cv_model, X_train_lr, y_train,
    groups=train_df['stay_id']
)
display(lr_cv)

Cross-Validation (5-fold GroupKFold):
  AUPRC: 0.2376 ± 0.0047
  AUROC: 0.7570 ± 0.0033


,fold,auprc,auroc
0,1,0.235459,0.759715
1,2,0.238619,0.757298
2,3,0.242216,0.758318
3,4,0.230576,0.751168
4,5,0.241107,0.758250


In [8]:
# Cross-validate XGBoost
xgb_cv_model = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, eval_metric='aucpr',
    random_state=42, n_jobs=-1
)
xgb_cv = cross_validate_model(
    xgb_cv_model, X_train_xgb, y_train,
    groups=train_df['stay_id']
)
display(xgb_cv)

Cross-Validation (5-fold GroupKFold):
  AUPRC: 0.3895 ± 0.0075
  AUROC: 0.8103 ± 0.0032


,fold,auprc,auroc
0,1,0.389819,0.812798
1,2,0.391415,0.813308
2,3,0.390627,0.810438
3,4,0.377384,0.805198
4,5,0.398095,0.809780


## 3.8 Model Comparison (Default Parameters, Test Set)

In [9]:
comparison = compare_models({
    'Logistic Regression': lr_metrics,
    'XGBoost': xgb_metrics,
})
display(comparison)

,AUPRC,AUROC,Precision,Recall,F1
Model,,,,,
Logistic Regression,0.2351,0.7546,0.1784,0.7507,0.2883
XGBoost,0.3967,0.8121,0.2267,0.7199,0.3448
